# DORA Metrics for Failure Only

## Env

In [ ]:
print("Kernel is working.")

In [ ]:
from dash import dcc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

## Data Ingestion

**Output:** pandas dataframe df_fail

In [ ]:
# Useful reference: <https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html>
raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module2_project/doraview/data/json/change_fail.json'

# Try reading with explicit encoding and error handling
try:
    df_fail = pd.read_json(raw_file_path, encoding='utf-8', convert_dates=["detected_at"]).sort_values(by=["detected_at"])
    print("\nDataframe info:")
    print(df_fail.info())
    print("\nFirst 5 rows:")
    print(df_fail.head())
except Exception as e:
    print(f"Error: {str(e)}")

## Data Manipulation

**Output:** Updates to pandas dataframe *df_fail* for use by figures.
- New aggregated month datetime column.
- New Moving Average columns for plotting (SMA, EMA, CMA)

In [ ]:
df_fail["month"] = df_fail["detected_at"].dt.month

df_fail.head(100)

## Figures

### ChatGPT - Change Failure Rate (CFR)

**Best Graph Type:** Stacked Bar Chart (success vs. failed deployments)

**Why:**

*CFR is a percentage:*

- failed deployments ÷ total deployments

**Stacked bars clearly show:**

- Volume of deployments
- Proportion that failed vs. succeeded
- How stability changes over time
- Visual impact of process or team improvements

**Optional secondary views:**

- Donut/Pie Chart for a single period (e.g., this month)
- Line Chart for the CFR % trend over time

For dashboards, a percentage KPI tile + stacked bars is very common.

### Multi-Group Display

In [ ]:
df_fail_grouped = df_fail.groupby([
	"application_id",
	"failed",
	"month",
	])["month"].count().reset_index(name='count')

df_fail.head(100)

In [ ]:
# display dataframe as figure
fig_month_multi_bar = px.bar(
	data_frame=df_fail_grouped,
	title="Total Monthly Deployments",	# Label for the figure.
	x="month",							# Column for use on x-axis
	y="count",							# Column for use on y-axis
	color="failed",						# Column for use as colour differentiation
	# barmode="group",					# Group bars together
	# subtitle="All Applications"
)

fig_month_multi_bar.update_yaxes(
	title_text="Failure Count"		# Update title for y-axis
)

fig_month_multi_bar.update_xaxes(
	title_text="Deployment Month",		# Update title for x-axis
	tickvals=list(range(1,13)),			# Specify values of x-axis
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']	# Allocate text to tick values.
)

fig_month_multi_bar.show()